# SNR Hadronic Model Fitting: Pion Decay

This notebook fits the hadronic (pion decay) model to the combined Fermi-LAT + H.E.S.S. gamma-ray spectrum,
following the methodology from **Hnatyk et al. 2022 (MNRAS)**.

## Model Description
- **Physics**: Cosmic ray protons accelerated at SNR shock interact with ambient medium protons,
  producing neutral pions that decay into gamma-rays (π⁰ → γγ)
- **Proton spectrum**: Exponential Cutoff Power-Law (ECPL)
- **Parameters**: Normalization, spectral index Γ, cutoff energy E_cut
- **Target density**: n_H ~ 10 cm⁻³ (shell density)
- **Distance**: 12.5 kpc

## Reference
Hnatyk et al. 2022, MNRAS, Table 1:
- ECPL: Γ = 2.41 ± 0.03, E_cut = 185.2 ± 9.5 TeV, W_p = 5.12×10⁵⁰ erg

In [ ]:
import numpy as np
import astropy.units as u
from astropy.table import QTable
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import naima
from naima.models import ExponentialCutoffPowerLaw, PionDecay

# Import our custom modules
from spectrum_builder import build_spectrum_for_naima
from naima_models.snr_pion_decay import (
    snr_pion_decay_ecpl,
    snr_lnprior_ecpl,
    SNR_DEFAULTS,
    compute_proton_energy,
    get_initial_params_ecpl,
    get_labels_ecpl,
)

## 1. Load or Build Spectrum Data

Option A: Use synthetic spectrum from published parameters  
Option B: Load actual Fermi-LAT data from file (when available)

In [ ]:
# Build spectrum (use fermi_file=None for synthetic, or path to actual data)
FERMI_DATA_FILE = None  # Set to path when you have actual Fermi data

data = build_spectrum_for_naima(
    fermi_file=FERMI_DATA_FILE,
    energy_min=0.2 * u.GeV,  # 200 MeV lower bound
    n_fermi_bins=10,
    n_hess_bins=10,
    use_hawc=False,
)

print(f"Spectrum has {len(data)} energy bins")
print(f"Energy range: {data['energy'].min():.2f} - {data['energy'].max():.2f}")
data

In [ ]:
# Plot the input spectrum
fig, ax = plt.subplots(figsize=(10, 6))

E = data['energy'].to(u.GeV)
flux = data['flux'].to(u.Unit('1/(cm2 s GeV)'))
flux_err = data['flux_error'].to(u.Unit('1/(cm2 s GeV)'))

# E² dN/dE (SED)
sed = (E**2 * flux).to(u.Unit('GeV/(cm2 s)'))
sed_err = (E**2 * flux_err).to(u.Unit('GeV/(cm2 s)'))

ax.errorbar(E.value, sed.value, yerr=sed_err.value, fmt='o', 
            color='blue', ecolor='blue', capsize=3, label='Fermi-LAT + H.E.S.S.')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Energy [GeV]')
ax.set_ylabel(r'$E^2 dN/dE$ [GeV cm$^{-2}$ s$^{-1}$]')
ax.set_title('Input Gamma-Ray Spectrum')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Set Up MCMC Fitting

Parameters to fit:
1. `log10(N0)` - log10 of normalization at 1 TeV [1/eV]
2. `Γ` - proton spectral index
3. `log10(E_cut/TeV)` - log10 of cutoff energy

In [ ]:
# Initial parameters (near expected values from paper)
p0 = get_initial_params_ecpl()
print(f"Initial parameters: {p0}")

# Parameter labels for plots
labels = get_labels_ecpl()
print(f"Labels: {labels}")

# MCMC settings
NWALKERS = 32
NBURN = 100    # Burn-in steps (increase for production)
NRUN = 500     # Production steps (increase for production)

In [ ]:
# Run the MCMC sampler
sampler, pos = naima.run_sampler(
    data_table=data,
    p0=p0,
    labels=labels,
    model=snr_pion_decay_ecpl,
    prior=snr_lnprior_ecpl,
    nwalkers=NWALKERS,
    nburn=NBURN,
    nrun=NRUN,
    threads=4,
    prefit=True,
)

## 3. Analyze Results

In [ ]:
# Save diagnostic plots
output_prefix = "snr_pion_decay_ecpl"
naima.save_diagnostic_plots(output_prefix, sampler, sed=True)
naima.save_results_table(output_prefix, sampler)
print(f"Saved diagnostic plots and results to {output_prefix}_*")

In [ ]:
# Extract best-fit and uncertainties
chain = sampler.get_chain(flat=True)
logp = sampler.get_log_prob(flat=True)

# Maximum a posteriori (MAP) estimate
best_idx = np.nanargmax(logp)
pars_map = chain[best_idx]

# Percentile estimates
q16, q50, q84 = np.percentile(chain, [16, 50, 84], axis=0)

print("=" * 60)
print("Best-fit Parameters (ECPL Pion Decay Model)")
print("=" * 60)
for i, label in enumerate(labels):
    print(f"{label}: {q50[i]:.3f} (+{q84[i]-q50[i]:.3f} / -{q50[i]-q16[i]:.3f})")
print()
print(f"E_cut = {10**q50[2]:.1f} (+{10**q84[2]-10**q50[2]:.1f} / -{10**q50[2]-10**q16[2]:.1f}) TeV")

In [ ]:
# Compute total proton energy from posterior
n_samples = min(500, len(chain))
idx = np.random.choice(len(chain), size=n_samples, replace=False)

Wp_samples = []
for i in idx:
    try:
        Wp = compute_proton_energy(chain[i], model='ecpl', Emin=1*u.GeV)
        Wp_samples.append(Wp.to(u.erg).value)
    except:
        pass

Wp_samples = np.array(Wp_samples)
Wp_q16, Wp_q50, Wp_q84 = np.percentile(Wp_samples, [16, 50, 84])

print(f"\nTotal proton energy W_p (E > 1 GeV):")
print(f"  W_p = {Wp_q50:.2e} (+{Wp_q84-Wp_q50:.2e} / -{Wp_q50-Wp_q16:.2e}) erg")
print(f"\nCompare to Hnatyk et al. 2022: W_p = 5.12e50 erg")

In [ ]:
# Compute chi-squared
model_flux = snr_pion_decay_ecpl(pars_map, data)
residuals = (model_flux - data['flux']) / data['flux_error']
chi2 = np.sum(residuals.value**2)
ndf = len(data) - len(pars_map)

print(f"\nGoodness of fit:")
print(f"  χ² = {chi2:.2f}")
print(f"  ndf = {ndf}")
print(f"  χ²/ndf = {chi2/ndf:.2f}")

## 4. Plot Best-Fit SED

In [ ]:
# Create energy grid for model
E_plot = np.logspace(-1, 5, 200) * u.GeV
grid = QTable({'energy': E_plot})

# Calculate model at best-fit
model_flux_plot = snr_pion_decay_ecpl(pars_map, grid)
model_sed = (E_plot**2 * model_flux_plot).to(u.Unit('GeV/(cm2 s)'))

# Data SED
data_E = data['energy'].to(u.GeV)
data_sed = (data_E**2 * data['flux']).to(u.Unit('GeV/(cm2 s)'))
data_sed_err = (data_E**2 * data['flux_error']).to(u.Unit('GeV/(cm2 s)'))

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

ax.errorbar(data_E.value, data_sed.value, yerr=data_sed_err.value,
            fmt='o', color='red', ecolor='red', capsize=3,
            label='Fermi-LAT + H.E.S.S. data', zorder=10)

ax.plot(E_plot.value, model_sed.value, 'b-', lw=2,
        label=f'Pion Decay (Γ={q50[1]:.2f}, E_cut={10**q50[2]:.0f} TeV)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.1, 1e5)
ax.set_ylim(1e-13, 1e-8)
ax.set_xlabel('Photon Energy [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2 dN/dE$ [GeV cm$^{-2}$ s$^{-1}$]', fontsize=12)
ax.set_title('SNR Hadronic Model: Pion Decay (ECPL Proton Spectrum)', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, which='both', ls=':', alpha=0.5)

plt.tight_layout()
plt.savefig('snr_pion_decay_sed.png', dpi=150)
plt.show()

print("Saved SED plot to snr_pion_decay_sed.png")

## 5. Summary Table (like Table 1 in paper)

In [ ]:
# Build summary table
summary_rows = [
    ('E_cr,min [GeV] (fixed)', 1.0, '-', '-'),
    ('E_cr,max [GeV] (fixed)', 1e6, '-', '-'),
    ('n_H [cm⁻³] (fixed)', SNR_DEFAULTS['nH'].value, '-', '-'),
    ('Distance [kpc] (fixed)', SNR_DEFAULTS['distance'].value, '-', '-'),
    ('log₁₀(N₀/eV⁻¹)', f'{q50[0]:.2f}', f'+{q84[0]-q50[0]:.2f}', f'-{q50[0]-q16[0]:.2f}'),
    ('Γ_p', f'{q50[1]:.2f}', f'+{q84[1]-q50[1]:.2f}', f'-{q50[1]-q16[1]:.2f}'),
    ('E_cut [TeV]', f'{10**q50[2]:.1f}', f'+{10**q84[2]-10**q50[2]:.1f}', f'-{10**q50[2]-10**q16[2]:.1f}'),
    ('W_p (>1 GeV) [erg]', f'{Wp_q50:.2e}', f'+{Wp_q84-Wp_q50:.2e}', f'-{Wp_q50-Wp_q16:.2e}'),
    ('χ²/ndf', f'{chi2/ndf:.2f}', '-', '-'),
]

print("\n" + "=" * 70)
print("Summary: SNR ECPL Pion Decay Model")
print("=" * 70)
print(f"{'Parameter':<25} {'Value':>15} {'Upper':>12} {'Lower':>12}")
print("-" * 70)
for row in summary_rows:
    print(f"{row[0]:<25} {row[1]:>15} {row[2]:>12} {row[3]:>12}")
print("=" * 70)